## Quick analysis of podcast reviews

This notebook loads podcast reviews from a tab-separated file (like the ones produced by the Apple Store scraping notebook) and runs a few quick analyses of the star ratings. You'll see a preview of the data in table format, followed by several different views of the same `rating` column, each highlighting something the others don't.
The original Python code was written by Lars Keuris in 2023 and extended for teaching by Monika Barget. Originally run on the Maastricht DSRI, the code was adjusted for Pyodide in 2026.

**Note:** This notebook runs via Live Code, directly in your browser. You upload your own reviews file below (or paste its contents). Nothing is saved, and everything is gone when you close the tab. To save a visualisation you like, please take a screenshot!

## Import modules

The first run can take a moment while `pandas` and `matplotlib` are loaded into your browser.

In [ ]:
import micropip
await micropip.install(["pandas", "matplotlib"])

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # hides the harmless pandas/pyarrow notice

import io
import pandas as pd
import matplotlib.pyplot as plt
from pyodide.ffi import create_proxy
from js import document
from IPython.display import display, HTML

print("Setup complete!")

## Upload your data

Run the cell below. A **Choose file** button appears. Select a reviews file from your own computer, for example the file produced by the Apple Store scraping notebook. The file must have a header row with at least the columns `date` and `rating`, and can be tab-separated (as produced by the scraping notebook) or comma-separated.

If no button appears or the upload fails, use the paste box in the next cell instead.

In [ ]:
df = None

def load_dataframe(text):
    """Read review data (tab- or comma-separated) from a string into the global DataFrame `df`."""
    global df
    lines = text.splitlines()
    if not lines:
        raise ValueError("the file is empty")
    sep = "\t" if "\t" in lines[0] else ","
    new_df = pd.read_csv(io.StringIO(text), sep=sep)
    new_df.columns = [str(c).strip() for c in new_df.columns]
    missing = [c for c in ("date", "rating") if c not in new_df.columns]
    if missing:
        raise ValueError(f"column(s) not found: {', '.join(missing)}. Found: {', '.join(new_df.columns)}")
    df = new_df
    return df

display(HTML('<div id="reviews-upload-box"><input type="file" id="reviews-upload" accept=".csv,.tsv,.txt"> '
             '<span id="reviews-upload-status">No file selected yet.</span></div>'))

file_input = document.getElementById("reviews-upload")
upload_status = document.getElementById("reviews-upload-status")

if file_input is None:
    # Fallback: add the controls to the page if the output was not yet in the DOM
    box = document.createElement("div")
    file_input = document.createElement("input")
    file_input.type = "file"
    file_input.accept = ".csv,.tsv,.txt"
    upload_status = document.createElement("span")
    upload_status.textContent = " No file selected yet."
    box.appendChild(file_input)
    box.appendChild(upload_status)
    document.body.appendChild(box)

async def _read_file(event):
    f = file_input.files.item(0)
    if f is None:
        upload_status.textContent = "No file selected yet."
        return
    try:
        load_dataframe(await f.text())
        upload_status.textContent = f"Loaded '{f.name}' with {len(df)} rows. Continue with the preview below."
    except Exception as e:
        upload_status.textContent = f"Could not read '{f.name}': {e}"

file_input.addEventListener("change", create_proxy(_read_file))

### Backup: paste the data

Only use this if the upload above does not work. Open your reviews file in any text editor, copy all of it (including the header row), paste it into the box below and click **Load pasted data**. Then continue with the preview.

In [ ]:
display(HTML(
    '<div>'
    '<textarea id="reviews-paste" rows="8" style="width:100%" '
    'placeholder="Paste the full contents of the reviews file here (tab- or comma-separated, with header row)"></textarea><br>'
    '<button id="reviews-load">Load pasted data</button> '
    '<span id="reviews-status"></span>'
    '</div>'
))

paste_box = document.getElementById("reviews-paste")
paste_button = document.getElementById("reviews-load")
paste_status = document.getElementById("reviews-status")

def _on_load_click(event):
    try:
        load_dataframe(paste_box.value)
        paste_status.textContent = f"Loaded {len(df)} rows. Continue with the preview below."
    except Exception as e:
        paste_status.textContent = f"Could not read the data: {e}"

if paste_button is None:
    print("The paste box could not be displayed in this environment.")
else:
    paste_button.addEventListener("click", create_proxy(_on_load_click))

## Preview the data

In [ ]:
if df is None:
    print("No data loaded yet. Upload a file in the upload step above, or use the paste box.")
else:
    display(df)

## Prepare the data

Parse the review dates and extract the year and month, so we can group reviews by time period.

In [ ]:
if df is not None:
    try:
        df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
    except ValueError:
        # other date format in your file: let pandas work it out
        try:
            df['date'] = pd.to_datetime(df['date'])
        except ValueError:
            raise ValueError("The 'date' column could not be read. Dates should look like "
                             "2024-03-29 15:04:05 (year-month-day).")
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    dfmonthly_median = df.groupby(["year", "month"]).median(numeric_only=True)
    dfmonthly_mean = df.groupby(["year", "month"]).mean(numeric_only=True)
    dfmonthly_count = df.groupby(["year", "month"]).size()

    print("Data prepared!")
else:
    print("No data loaded yet. Upload a file in the upload step above, or use the paste box.")

## Visualisation 1: Median rating per month

The visualisations shows how the typical (median) rating moved over time. Medians are robust to a handful of extreme reviews, but they can also hide how spread out opinions really were in a given month.

In [ ]:
if df is not None:
    dfmonthly_median['rating'].plot(title='Median rating per month', ylabel='Median rating', xlabel='Year, month')
    plt.show()

## Visualisation 2: Mean vs. median rating per month

Plotting the mean alongside the median shows where a small number of very high or very low ratings are pulling the average away from what most reviewers actually said. If the two lines track closely, ratings in that month were fairly consistent; where they diverge, a few outlier reviews are having an outsized effect on the mean.

In [ ]:
if df is not None:
    fig, ax = plt.subplots()
    dfmonthly_mean['rating'].plot(ax=ax, label='Mean', title='Mean vs. median rating per month')
    dfmonthly_median['rating'].plot(ax=ax, label='Median')
    ax.set_ylabel('Rating')
    ax.set_xlabel('Year, month')
    ax.legend()
    plt.show()

## Visualisation 3: Distribution of star ratings

App Store reviews are often bimodal (lots of 1-star and 5-star reviews, with fewer in between), which a single average or median completely hides. This chart shows how many reviews fall into each star rating overall.

In [ ]:
if df is not None:
    df['rating'].value_counts().sort_index().plot(
        kind='bar', title='Distribution of star ratings', xlabel='Star rating', ylabel='Number of reviews'
    )
    plt.show()

## Visualisation 4: Review volume per month

A rating trend is easier to interpret once you know how many reviews it's based on. A month with an average of 4.8 from 300 reviews means something quite different from a month with an average of 4.8 from 2 reviews.

In [ ]:
if df is not None:
    dfmonthly_count.plot(
        kind='bar', title='Number of reviews per month', xlabel='Year, month', ylabel='Number of reviews'
    )
    plt.show()

## Visualisation 5: Rating spread per year

A box plot shows the full spread of ratings within each year (the middle 50% in the box, the median as the line inside it, and outliers) rather than collapsing everything into a single number.

In [ ]:
if df is not None:
    df.boxplot(column='rating', by='year')
    plt.title('Rating spread per year')
    plt.suptitle('')
    plt.xlabel('Year')
    plt.ylabel('Rating')
    plt.show()